Create a chronological train and test split from the Phase 1 clean dataset. The cutoff date is inclusive for training.

## **Step 0: Setup**

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import sys
sys.path.append("..")

from src.model import (
    load_operating_rows,
    time_split
)

## **Step 1: Load and Split**

In [2]:
operating_rows = load_operating_rows()
train, test = time_split(operating_rows, cutoff_date="2018-09-30")
total_rows = len(operating_rows)

print(f"Operating rows: {total_rows:,}")
print()
for name, part in (("Train", train), ("Test", test)):
    share = len(part) / total_rows if total_rows else 0
    first_date = part["date"].min().date() if not part.empty else None
    last_date = part["date"].max().date() if not part.empty else None
    print(f"{name} rows: {len(part):,}")
    print(f"{name} share: {share:.1%}")
    print(f"{name} first date: {first_date}")
    print(f"{name} last date: {last_date}")
    print()

Operating rows: 8,465

Train rows: 7,152
Train share: 84.5%
Train first date: 2017-12-01
Train last date: 2018-09-29

Test rows: 1,313
Test share: 15.5%
Test first date: 2018-10-01
Test last date: 2018-11-30



## **Step 2: Split Assertions**

In [23]:
ROW_LENGTH = 24
EXPECTED_DATE = pd.Timestamp("2018-10-01")
EXPECTED_HOUR = 0
last_train_date = pd.Timestamp("2018-09-29")

# 1. Training ends before testing begins.
assert train["date"].max() < test["date"].min()

# 2. No date and hour row appears in both partitions.
train_keys = set(zip(train["date"], train["hour"]))
test_keys = set(zip(test["date"], test["hour"]))
assert train_keys.isdisjoint(test_keys)

# 3. Every operating row belongs to exactly one partition.
assert len(train) + len(test) == len(operating_rows)

# 4. The split meets the exact cutoff boundary.
assert train["date"].max() == last_train_date
assert test["date"].min() == EXPECTED_DATE

# 5. No date carries a time part.
time_part_rows = (operating_rows["date"] != operating_rows["date"].dt.normalize()).sum()
assert time_part_rows == 0, f"{time_part_rows} date values carry a time part."

# 6. The last train date holds 24 rows.
actual_len = len(train.loc[train["date"] == last_train_date])
assert actual_len == ROW_LENGTH, f"Row count mismatch: expected {ROW_LENGTH}, got {actual_len}."

# 7. The first test row is 2018-10-01 at hour 0.
first_test_row = test.iloc[0]
assert first_test_row["date"] == EXPECTED_DATE, f"Date mismatch: expected {EXPECTED_DATE}, got {first_test_row['date']}."
assert first_test_row["hour"] == EXPECTED_HOUR, f"Hour mismatch: expected {EXPECTED_HOUR}, got {first_test_row['hour']}."

print("All 7 time split assertions PASSED.")

All 7 time split assertions PASSED.


## **Step 3: Feature Inventory**

In [24]:
FEATURE_PLAN = {
    # column: (role, use, reason)
    "date": ("known to the app", "no",
             "The user picks the date. The raw datetime stays out of the model. The derived columns carry its information."),
    "rented_bike_count": ("target", "target",
             "The number of rentals in one hour. The model predicts this column."),
    "hour": ("known to the app", "yes",
             "The user picks the hour."),
    "temperature": ("typed by the user (plan B)", "yes",
             "The user types a forecast. Training holds measured values. Forecast error reaches the model."),
    "humidity": ("typed by the user (plan B)", "yes",
             "The user types a forecast. Training holds measured values. Forecast error reaches the model."),
    "wind_speed": ("typed by the user (plan B)", "yes",
             "The user types a forecast. Training holds measured values. Forecast error reaches the model."),
    "rainfall": ("typed by the user (plan B)", "yes",
             "The user types a forecast. Training holds measured values. Forecast error reaches the model."),
    "snowfall": ("typed by the user (plan B)", "yes",
             "The user types a forecast. Training holds measured values. Forecast error reaches the model."),
    "visibility": ("not available to the app", "no",
             "A user does not know visibility, in units of 10 m, for a future hour. Plan B has no input field for it."),
    "dew_point_temperature": ("not available to the app", "no",
             "Computed from temperature and humidity. A user does not type it. Step 3 checks the overlap with temperature."),
    "solar_radiation": ("not available to the app", "no",
             "Depends on sky cover and the hour. A user does not know the value for a future hour."),
    "seasons": ("derived from date or hour", "review",
             "Set by month. Month holds more detail. Step 3 tests whether season adds anything."),
    "holiday": ("known to the app", "yes",
             "A public holiday calendar gives the flag before the prediction."),
    "functioning_day": ("constant", "no",
             "Every row holds Yes after the closed days leave. The column carries no information."),
    "year": ("derived from date or hour", "no",
             "Train holds 2017 (December only) and 2018. A future date holds a year the model never saw."),
    "month": ("derived from date or hour", "yes",
             "The calendar month. The value repeats every year."),
    "weekday": ("derived from date or hour", "yes",
             "The day of the week as a number, Monday 0 to Sunday 6."),
    "day_name": ("derived from date or hour", "no",
             "A text copy of weekday. The models in this project read numbers."),
    "is_weekend": ("derived from date or hour", "yes",
             "A flag for Saturday and Sunday."),
    "day_position": ("derived from date or hour", "review",
             "The position in a Friday to Thursday week. Holds the same information as weekday. Step 3 decides which one stays."),
    "week_number": ("derived from date or hour", "no",
             "Counts weeks from 1 Dec 2017. Train holds weeks 1 to 44. Test holds weeks 44 to 53. The model never sees weeks 45 to 53."),
}

missing = set(operating_rows.columns) - set(FEATURE_PLAN)
extra = set(FEATURE_PLAN) - set(operating_rows.columns)
assert not missing and not extra, f"Missing: {sorted(missing)}. Extra: {sorted(extra)}."

column_roles = pd.DataFrame(
    [
        {"column": col, "dtype": operating_rows[col].dtype, "role": role, "use": use, "reason": reason}
        for col, (role, use, reason) in FEATURE_PLAN.items()
    ]
)
pd.set_option("display.max_colwidth", None)
display(column_roles)

,column,dtype,role,use,reason
0,date,datetime64[ns],known to the app,no,The user picks the date. The raw datetime stays out of the model. The derived columns carry its information.
1,rented_bike_count,int64,target,target,The number of rentals in one hour. The model predicts this column.
2,hour,int64,known to the app,yes,The user picks the hour.
3,temperature,float64,typed by the user (plan B),yes,The user types a forecast. Training holds measured values. Forecast error reaches the model.
4,humidity,int64,typed by the user (plan B),yes,The user types a forecast. Training holds measured values. Forecast error reaches the model.
5,wind_speed,float64,typed by the user (plan B),yes,The user types a forecast. Training holds measured values. Forecast error reaches the model.
6,rainfall,float64,typed by the user (plan B),yes,The user types a forecast. Training holds measured values. Forecast error reaches the model.
7,snowfall,float64,typed by the user (plan B),yes,The user types a forecast. Training holds measured values. Forecast error reaches the model.
8,visibility,int64,not available to the app,no,"A user does not know visibility, in units of 10 m, for a future hour. Plan B has no input field for it."
9,dew_point_temperature,float64,not available to the app,no,Computed from temperature and humidity. A user does not type it. Step 3 checks the overlap with temperature.
